# RT Notebook 13 v2: Ordered Chunked Topological Invariants

This supersedes the v1 Colab execution scaffold. It is ordered, chunked, and resumable. Run cells from top to bottom. If Colab resets, rerun the bootstrap cell, then continue from the chunk runner; completed chunks are skipped.

Claim ceiling before governed output review: `C1_SPECIFICATION_ONLY`.


## 1. Upload or Mount Notebook 12 Zip

Put `rt_notebook_12_outputs_results.zip` at one of these paths, or edit `NB12_ZIP` in the bootstrap cell:

- `/content/rt_notebook_12_outputs_results.zip`
- `/content/drive/MyDrive/rt_notebook_12_outputs_results.zip`
- local repo path when running outside Colab


In [ ]:
# Optional in Colab: uncomment to upload the Notebook 12 zip manually.
# from google.colab import files
# files.upload()


## 2. Bootstrap: Imports, Paths, Notebook 12 Graph Builder, Invariants

This cell intentionally defines all runtime dependencies together. Re-run this cell after any Colab reset.


In [ ]:
from __future__ import annotations

import hashlib, itertools, json, math, os, zipfile
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Set, Tuple

import networkx as nx
import numpy as np
import pandas as pd

SEED = 130013
SPEC_ID = "NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002"
ZIP_CANDIDATES = [
    Path('/content/rt_notebook_12_outputs_results.zip'),
    Path('/content/drive/MyDrive/rt_notebook_12_outputs_results.zip'),
    Path('rt_notebook_12_outputs_results.zip'),
    Path(r'D:/projects/New folder/rt_notebook_12_outputs_results.zip'),
]
NB12_ZIP = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if NB12_ZIP is None:
    raise FileNotFoundError('Could not find rt_notebook_12_outputs_results.zip. Upload it to /content or mount Drive, then rerun this cell.')

OUTPUT_DIR = Path('/content') / SPEC_ID if Path('/content').exists() else Path('departments/colab/results') / SPEC_ID
CHUNK_DIR = OUTPUT_DIR / 'chunks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 250

MECHANISM_NAMES = [
    'residue_persistence', 'kernel_deformation', 'scheduling_effects',
    'termination_policy', 'admissibility_restrictions', 'suffix_interactions'
]

@dataclass(frozen=True)
class MechanismMask:
    residue_persistence: bool = False
    kernel_deformation: bool = False
    scheduling_effects: bool = False
    termination_policy: bool = False
    admissibility_restrictions: bool = False
    suffix_interactions: bool = False
    def active_count(self) -> int:
        return sum(bool(getattr(self, name)) for name in MECHANISM_NAMES)
    def bitstring(self) -> str:
        return ''.join('1' if getattr(self, name) else '0' for name in MECHANISM_NAMES)
    @classmethod
    def from_bits(cls, bits: Sequence[int]) -> 'MechanismMask':
        return cls(**{name: bool(bit) for name, bit in zip(MECHANISM_NAMES, bits)})

NEUTRAL_MASK = MechanismMask()

@dataclass(frozen=True, order=True)
class State:
    value: int
    residue: int
    phase: int
    suffix: int
    depth: int

@dataclass(frozen=True)
class SystemConfig:
    state_modulus: int = 5
    max_depth: int = 6
    branch_width: int = 2
    initial_value: int = 0
    initial_suffix: int = 0
    schedule_seed: int = 0
    mask: MechanismMask = NEUTRAL_MASK
    def stable_id(self) -> str:
        payload = json.dumps({
            'state_modulus': self.state_modulus, 'max_depth': self.max_depth,
            'branch_width': self.branch_width, 'initial_value': self.initial_value,
            'initial_suffix': self.initial_suffix, 'schedule_seed': self.schedule_seed,
            'mask': asdict(self.mask)
        }, sort_keys=True).encode()
        return hashlib.sha256(payload).hexdigest()[:16]

def config_from_nb12_row(row: Mapping[str, Any]) -> SystemConfig:
    missing = [c for c in ['state_modulus','max_depth','branch_width','initial_value','initial_suffix','schedule_seed', *MECHANISM_NAMES] if c not in row]
    if missing:
        raise KeyError(f'Notebook 12 row missing required columns: {missing}')
    mask = MechanismMask(**{name: bool(row[name]) for name in MECHANISM_NAMES})
    return SystemConfig(
        state_modulus=int(row['state_modulus']), max_depth=int(row['max_depth']),
        branch_width=int(row['branch_width']), initial_value=int(row['initial_value']),
        initial_suffix=int(row['initial_suffix']), schedule_seed=int(row['schedule_seed']), mask=mask
    )

def initial_state(config: SystemConfig) -> State:
    return State(config.initial_value % config.state_modulus, 0, 0, config.initial_suffix % config.state_modulus, 0)

def should_terminate(state: State, config: SystemConfig) -> bool:
    if state.depth >= config.max_depth:
        return True
    if config.mask.termination_policy:
        return state.depth >= 2 and ((state.value + state.residue + state.phase) % config.state_modulus == 0)
    return False

def transition_candidates(state: State, config: SystemConfig) -> List[State]:
    if should_terminate(state, config):
        return []
    candidates = []
    for branch_index in range(config.branch_width):
        residue_term = state.residue if config.mask.residue_persistence else 0
        phase_term = state.phase if config.mask.kernel_deformation else 0
        suffix_term = state.suffix if config.mask.suffix_interactions else 0
        schedule_term = 0
        if config.mask.scheduling_effects:
            schedule_term = (config.schedule_seed + state.depth + branch_index * (state.value + 1)) % config.state_modulus
        next_value = (state.value + 1 + branch_index + residue_term + phase_term + suffix_term + schedule_term) % config.state_modulus
        next_residue = (state.residue + state.value + branch_index + 1) % config.state_modulus if config.mask.residue_persistence else 0
        next_phase = (state.phase + state.value + 2 * branch_index + 1) % config.state_modulus if config.mask.kernel_deformation else 0
        next_suffix = (state.suffix + state.value + branch_index + 1) % config.state_modulus if config.mask.suffix_interactions else state.suffix
        candidate = State(next_value, next_residue, next_phase, next_suffix, state.depth + 1)
        if config.mask.admissibility_restrictions:
            if (candidate.value + candidate.residue + candidate.suffix + branch_index) % 3 == 0:
                continue
        candidates.append(candidate)
    return list(dict.fromkeys(candidates))

def build_continuation_graph(config: SystemConfig) -> nx.DiGraph:
    graph = nx.DiGraph()
    root = initial_state(config)
    graph.add_node(root)
    frontier, expanded = [root], set()
    while frontier:
        state = frontier.pop(0)
        if state in expanded:
            continue
        expanded.add(state)
        for successor in transition_candidates(state, config):
            graph.add_edge(state, successor)
            if successor not in expanded:
                frontier.append(successor)
    graph.graph['config_id'] = config.stable_id()
    graph.graph['mask'] = config.mask.bitstring()
    return graph

def build_graph_from_nb12_row(row: Mapping[str, Any]) -> nx.DiGraph:
    return build_continuation_graph(config_from_nb12_row(row))

def node_depths_from_roots(graph: nx.DiGraph) -> Dict[Any, int]:
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    depths, queue = {}, deque((root, 0) for root in roots)
    while queue:
        node, depth = queue.popleft()
        if node in depths and depths[node] <= depth:
            continue
        depths[node] = depth
        for succ in graph.successors(node):
            queue.append((succ, depth + 1))
    return depths

def root_successor_pairs(graph: nx.DiGraph) -> List[Tuple[Any, Any]]:
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    if not roots:
        return []
    root = min(roots, key=lambda x: getattr(x, 'depth', 0))
    successors = list(graph.successors(root))
    return list(itertools.combinations(successors, 2))

def exact_or_bounded_width(dag: nx.DiGraph, max_antichains: int = 20000) -> Tuple[int, bool]:
    width, exact = 0, True
    for i, antichain in enumerate(nx.antichains(dag)):
        if i >= max_antichains:
            exact = False
            break
        width = max(width, len(antichain))
    return width, exact

def earliest_common_depth(graph: nx.DiGraph, left: Any, right: Any, depths: Mapping[Any, int]) -> Optional[int]:
    common = ({left} | nx.descendants(graph, left)) & ({right} | nx.descendants(graph, right))
    return None if not common else min(depths.get(node, math.inf) for node in common)

def topological_invariants(graph: nx.DiGraph) -> Dict[str, Any]:
    depths = node_depths_from_roots(graph)
    closures = {node: frozenset({node} | nx.descendants(graph, node)) for node in graph.nodes}
    unique_closures = set(closures.values())
    condensation = nx.condensation(graph)
    undirected = graph.to_undirected(as_view=False)
    articulation_points = list(nx.articulation_points(undirected)) if undirected.number_of_nodes() else []
    bridges = list(nx.bridges(undirected)) if undirected.number_of_nodes() else []
    width, width_exact = exact_or_bounded_width(condensation)
    pairs = root_successor_pairs(graph)
    merge_depths = [earliest_common_depth(graph, a, b, depths) for a, b in pairs]
    observed = [d for d in merge_depths if d is not None]
    separated = sum(1 for d in merge_depths if d is None)
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    root = min(roots, key=lambda x: getattr(x, 'depth', 0)) if roots else None
    dom_count, dom_depth = 0, 0
    if root is not None:
        try:
            dom = nx.immediate_dominators(graph, root)
            dom_count = len(dom)
            dom_tree = nx.DiGraph((parent, child) for child, parent in dom.items() if child != parent)
            dom_depth = max([0] + [nx.shortest_path_length(dom_tree, root, n) for n in dom_tree.nodes if nx.has_path(dom_tree, root, n)])
        except Exception:
            dom_count, dom_depth = -1, -1
    sink_sccs = [n for n in condensation.nodes if condensation.out_degree(n) == 0]
    source_sccs = [n for n in condensation.nodes if condensation.in_degree(n) == 0]
    return {
        'node_count': graph.number_of_nodes(), 'edge_count': graph.number_of_edges(),
        'reachability_closure_count': len(unique_closures),
        'reachability_collision_count': graph.number_of_nodes() - len(unique_closures),
        'closure_min_size': min((len(c) for c in unique_closures), default=0),
        'closure_max_size': max((len(c) for c in unique_closures), default=0),
        'scc_count': nx.number_strongly_connected_components(graph),
        'condensation_node_count': condensation.number_of_nodes(),
        'condensation_edge_count': condensation.number_of_edges(),
        'condensation_source_count': len(source_sccs), 'condensation_sink_count': len(sink_sccs),
        'partial_order_width': width, 'partial_order_width_exact': width_exact,
        'terminal_basin_count': len(sink_sccs),
        'articulation_point_count': len(articulation_points),
        'articulation_min_depth': min((depths.get(n, math.inf) for n in articulation_points), default=-1),
        'articulation_max_depth': max((depths.get(n, -1) for n in articulation_points), default=-1),
        'bridge_count': len(bridges), 'root_branch_pair_count': len(pairs),
        'irreversible_separated_pair_count': separated, 'has_irreversible_separation': separated > 0,
        'merge_depth_min': min(observed, default=-1),
        'merge_depth_mean': sum(observed) / len(observed) if observed else -1,
        'merge_depth_max': max(observed, default=-1),
        'dominance_tree_node_count': dom_count, 'dominance_tree_max_depth': dom_depth,
        'closure_lattice_node_count': len(unique_closures)
    }

print('Using NB12 zip:', NB12_ZIP)
print('Output dir:', OUTPUT_DIR)


## 3. Load Notebook 12 Tables


In [ ]:
with zipfile.ZipFile(NB12_ZIP) as zf:
    with zf.open('factorial_results.parquet') as f:
        nb12 = pd.read_parquet(f)
    with zf.open('manifest.json') as f:
        nb12_manifest = json.load(f)

required = ['config_id','geometry','J','state_modulus','max_depth','branch_width','initial_value','initial_suffix','schedule_seed', *MECHANISM_NAMES]
missing = [c for c in required if c not in nb12.columns]
if missing:
    raise KeyError(f'factorial_results.parquet missing required columns: {missing}')
print('Loaded rows:', len(nb12))
print('Columns OK')
nb12[['config_id','geometry','J']].head()


## 4. Run One Chunk

Run this cell repeatedly until it prints that all chunks are complete. It saves each chunk before moving on.


In [ ]:
def process_chunk(start: int, stop: int) -> Dict[str, Any]:
    out_path = CHUNK_DIR / f'chunk_{start:05d}_{stop:05d}.parquet'
    skip_path = CHUNK_DIR / f'chunk_{start:05d}_{stop:05d}_skips.json'
    if out_path.exists():
        return {'status': 'already_done', 'path': str(out_path), 'start': start, 'stop': stop}
    rows, skips = [], []
    for _, row in nb12.iloc[start:stop].iterrows():
        try:
            graph = build_graph_from_nb12_row(row)
            inv = topological_invariants(graph)
            inv.update({'config_id': row['config_id'], 'geometry': row['geometry'], 'J': float(row['J'])})
            rows.append(inv)
        except Exception as exc:
            skips.append({'config_id': row.get('config_id'), 'error': repr(exc)})
    pd.DataFrame(rows).to_parquet(out_path, index=False)
    skip_path.write_text(json.dumps(skips, indent=2), encoding='utf-8')
    return {'status': 'written', 'path': str(out_path), 'rows': len(rows), 'skips': len(skips), 'start': start, 'stop': stop}

chunk_ranges = [(i, min(i + CHUNK_SIZE, len(nb12))) for i in range(0, len(nb12), CHUNK_SIZE)]
pending = [(a, b) for a, b in chunk_ranges if not (CHUNK_DIR / f'chunk_{a:05d}_{b:05d}.parquet').exists()]
if not pending:
    print('All chunks complete:', len(chunk_ranges))
else:
    result = process_chunk(*pending[0])
    print(result)
    print('Remaining chunks after this cell:', max(0, len(pending) - 1))


## 5. Aggregate Checkpoints

Run after all chunks are complete. This creates `invariants`, so later cells no longer depend on a fragile all-at-once loop.


In [ ]:
chunk_files = sorted(CHUNK_DIR.glob('chunk_*.parquet'))
if not chunk_files:
    raise RuntimeError('No chunk files found. Run the chunk cell first.')
invariants = pd.concat([pd.read_parquet(p) for p in chunk_files], ignore_index=True)
invariants.to_parquet(OUTPUT_DIR / 'topological_invariants.parquet', index=False)
skip_files = sorted(CHUNK_DIR.glob('chunk_*_skips.json'))
skips = []
for p in skip_files:
    skips.extend(json.loads(p.read_text(encoding='utf-8')))
(OUTPUT_DIR / 'skipped_configurations.json').write_text(json.dumps(skips, indent=2), encoding='utf-8')
print('Aggregated invariant rows:', len(invariants))
print('Total skips:', len(skips))
invariants.head()


## 6. Classify Geometry from Topology-Only Features


In [ ]:
EXCLUDED = {'config_id', 'geometry', 'J', 'mask_bits', 'active_mechanism_count', *MECHANISM_NAMES}
feature_columns = [c for c in invariants.columns if c not in EXCLUDED and pd.api.types.is_numeric_dtype(invariants[c])]
report = {'spec_id': SPEC_ID, 'feature_columns': feature_columns, 'claim_ceiling': 'C2_CANDIDATE_AFTER_GOVERNED_REVIEW'}
try:
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import balanced_accuracy_score, classification_report
    from sklearn.model_selection import train_test_split
    X = invariants[feature_columns].fillna(-1)
    y = invariants['geometry']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)
    model = RandomForestClassifier(n_estimators=120, random_state=SEED, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    report.update({
        'status': 'EXECUTED_CLASSIFIER',
        'dummy_balanced_accuracy': float(balanced_accuracy_score(y_test, dummy.predict(X_test))),
        'topology_balanced_accuracy': float(balanced_accuracy_score(y_test, model.predict(X_test))),
        'classification_report': classification_report(y_test, model.predict(X_test), output_dict=True)
    })
except Exception as exc:
    report.update({'status': 'INCONCLUSIVE_DEPENDENCY_OR_EXECUTION_FAILURE', 'error': repr(exc)})
(OUTPUT_DIR / 'topology_classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
report


## 7. Counterexamples and Manifest


In [ ]:
signature_columns = [c for c in feature_columns if c not in {'node_count', 'edge_count'}]
counterexamples = []
for signature, group in invariants.groupby(signature_columns, dropna=False):
    labels = sorted(group['geometry'].astype(str).unique())
    if len(labels) > 1 or float(group['J'].max()) != float(group['J'].min()):
        sig_values = signature if isinstance(signature, tuple) else (signature,)
        counterexamples.append({
            'signature': dict(zip(signature_columns, [str(v) for v in sig_values])),
            'row_count': int(len(group)),
            'geometry_labels': labels,
            'J_min': float(group['J'].min()),
            'J_max': float(group['J'].max()),
            'example_config_ids': group['config_id'].head(10).tolist()
        })
(OUTPUT_DIR / 'topology_counterexamples.json').write_text(json.dumps(counterexamples, indent=2), encoding='utf-8')
manifest = {
    'notebook': 'RT Notebook 13 v2',
    'title': 'Ordered Chunked Topological Invariants of Continuation Geometry',
    'spec_id': SPEC_ID,
    'seed': SEED,
    'source_archive': str(NB12_ZIP),
    'rows_loaded': int(len(nb12)),
    'invariant_rows': int(len(invariants)),
    'skip_count': int(len(skips)),
    'chunk_size': CHUNK_SIZE,
    'claim_ceiling': 'C1 before governed output induction; C2 candidate only after result zip registration and review',
    'interpretation_constraint': 'Bounded to regenerated Notebook 12 continuation-graph domain; no external physical validation.'
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Counterexamples:', len(counterexamples))
print('Output dir:', OUTPUT_DIR)
manifest
